# ACCESS-AIS3 -- SSA floating-ice rheology inversion

## Imports & helper functions

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Configure options

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'


all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
]

# Define steps to run (this notebook is scoped to this group)
steps = ['ssa_rheology_floating_inv_sensit', 'ssa_rheology_floating_inv_lcurve']

## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

## Initialise data catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## SSA Rheology Inversion Sensitivity - Floating Ice

In [ ]:
if 'ssa_rheology_floating_inv_sensit' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA RHEOLOGY INVERSION SENSITIVITY - FLOATING ICE"          )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Defining inversion parameters...")
    md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
    md.inversion.iscontrol = 1
    md.inversion.control_parameters = ['MaterialsRheologyBbar']
    md.inversion.min_parameters = pyissm.tools.materials.cuffey(273.15 - 0) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.max_parameters = pyissm.tools.materials.cuffey(273.15 - 70) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.maxsteps = 500
    md.inversion.maxiter = 200

    # Remove icebergs
    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print('-- Extracting floating ice only...')
    mask = (md.mask.ocean_levelset < 0) & (md.mask.ice_levelset < 0) # Floating ice
    mds = md.extract(mask)

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    print(f"-- Setting-up coefficient grid...")
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [1, 10, 100, 1000],
         103: [1, 10, 100, 1000]})

    print(f"-- Defining mask to exclude 0 velocity...")
    mask = (mds.inversion.vel_obs > 0)

    if save:
        print(f"-- Loading inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_sensit',
            run = False,
            load_only = True,
            global_mask = mask)

        print(f"-- Processing inversion parameter sensitivity...")
        diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir=f'{model_dir}/AIS3_ssa_rheology_floating_inv_sensit/')

        diagnostics_norm = pyissm.inversion.sensitivity.normalize_diagnostics(diagnostics,
                                                                             columns = ['vel_rmse', 'mean_gradient_magnitude'])
        diagnostics_norm['overall'] = diagnostics_norm['vel_rmse_norm'] + diagnostics_norm['mean_gradient_magnitude_norm']

        fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize = (15, 8), constrained_layout = True)
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax1, value = 'vel_rmse')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax2, value = 'ratio_101_103')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax3, value = 'mean_gradient_magnitude')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax4, value = 'cost_total')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics, x = 'cf101', y = 'cf103', ax = ax5, value = 'positive_residual_fraction')
        pyissm.inversion.plot.plot_sensitivity_heatmap(diagnostics_norm, x = 'cf101', y = 'cf103', ax = ax6, value = 'overall')
        plt.savefig(f'{model_dir}/AIS3_ssa_rheology_floating_inv_sensit/diagnostic_heatmaps.png')

        best_row = diagnostics_norm.loc[diagnostics_norm['overall'].idxmax()]
        print(f"The best run_id is: {best_row['run_id']}. This uses the following coefficient values:")
        print(best_row.filter(regex=r'^cf'))

    else:
        print(f"-- Running inversion parameter sensitivity...")
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_sensit',
            run = True,
            load_only = False,
            global_mask = mask)

## SSA Rheology Inversion L-Curve - Floating Ice

In [ ]:
if 'ssa_rheology_floating_inv_lcurve' in steps:

    print("-------------------------------------------------------------")
    print(f" SSA RHEOLOGY INVERSION L-CURVE - FLOATING ICE"              )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Define general control parameters...")
    md.inversion.iscontrol = 1
    md.verbose.solution = 0
    md.verbose.qmu = 0
    md.verbose.control = 1

    print(f"-- Defining inversion parameters...")
    md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
    md.inversion.iscontrol = 1
    md.inversion.control_parameters = ['MaterialsRheologyBbar']
    md.inversion.min_parameters = pyissm.tools.materials.cuffey(273.15 - 0) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.max_parameters = pyissm.tools.materials.cuffey(273.15 - 70) * np.ones((md.mesh.numberofvertices, ))
    md.inversion.maxsteps = 500
    md.inversion.maxiter = 200

    # Remove icebergs
    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print('-- Extracting floating ice only...')
    mask = (md.mask.ocean_levelset < 0) & (md.mask.ice_levelset < 0) # Floating ice
    mds = md.extract(mask)

    print(f"-- Assigning cluster and updating settings...")
    mds.cluster = cluster
    mds.settings.waitonlock = 0

    print(f"-- Setting-up coefficient grid...")
    # Use preferred 101/103 coefficients from sensitivity step
    param_grid = pyissm.inversion.sensitivity.build_parameter_grid(
        {101: [1],
         103: [10],
         502: [1e-20, 1e-19, 1e-18, 1e-17, 1e-16, 1e-15, 1e-14, 1e-13, 1e-12]})

    print(f"-- Defining mask to exclude 0 velocity...")
    mask = (mds.inversion.vel_obs > 0)

    if save:
        print(f"-- Loading inversion parameter sensitivity...")
        # Only mask 101 and 103 -- no mask on regularisation.
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve',
            run = False,
            load_only = True,
            coeff_masks = {101: mask,
                           103: mask})

        print(f"-- Processing inversion parameter sensitivity...")
        diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(manifest, output_dir=f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/')

        fig, ax = pyissm.inversion.plot.plot_lcurve(diagnostics)
        ax.set_title('Floating ice rheology inversion - L-curve analysis')
        plt.savefig(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/lcurve.png')

    else:
        print(f"-- Running inversion parameter sensitivity...")
        # Only mask 101 and 103 -- no mask on regularisation.
        manifest = pyissm.inversion.sensitivity.parameter_sensitivity(
            mds,
            param_grid,
            output_dir = f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve',
            run = True,
            load_only = False,
            coeff_masks = {101: mask,
                           103: mask})